In [ ]:
import os
import pickle
import re
from astroquery.utils.tap.core import TapPlus
from astroquery.ipac.ned import Ned

# Función para sanear nombres de archivo
def sanitize(name):
    # Solo letras, números, guión bajo o guión medio; el resto → '_'
    return re.sub(r'[^A-Za-z0-9_-]', '_', name)

# 0. Preparar directorio de salida
output_dir = "spectrums NASA"
os.makedirs(output_dir, exist_ok=True)

# 1. Cargar o inicializar z_init
pickle_path = 'extra/redshift_init_ned.pickle'
try:
    with open(pickle_path, 'rb') as f:
        z_init = pickle.load(f)
except FileNotFoundError:
    z_init = 0.0

# 2. Obtener objetos con espectros
batch_size = 10000
ned_tap = TapPlus(url="https://ned.ipac.caltech.edu/tap")
adql = f"""
SELECT TOP {batch_size}
    prefname, z
FROM NEDTAP.objdir
WHERE
    z >= {z_init:.7f}
    AND z < 3.2
    AND n_spectra > 0
ORDER BY z
"""
table = ned_tap.launch_job(adql).get_results().to_pandas()

# 3. Descargar FITS con nombre sanitizado y sufijo de redshift
for _, row in table.iterrows():
    name, z = row['prefname'], row['z']
    try:
        spectra = Ned.get_spectra(name, show_progress=False)
    except Exception:
        continue
    base = sanitize(name)                # nombre limpio
    for idx, hdulist in enumerate(spectra, start=1):
        # Índice antes, redshift al final tras 'z'
        filename = f"{base}_{idx}_z{z:.5f}.fits"
        path     = os.path.join(output_dir, filename)
        if not os.path.exists(path):
            hdulist.writeto(path, overwrite=True)
            print(f"Guardado: {filename}")

# 4. Actualizar z_init
if not table.empty:
    new_z_init = float(table['z'].iloc[-1]) + 0.000002
    with open(pickle_path, 'wb') as f:
        pickle.dump(new_z_init, f)
    print(f"z_init actualizado a {new_z_init:.7f}")

Guardado: ESO_244-_G_046_________________1_z0.00000.fits
Guardado: UM_424_________________________1_z0.00000.fits
Guardado: NGC_4236_______________________1_z0.00000.fits
Guardado: ESO_517-__005__________________1_z0.00000.fits
Guardado: SDSS_J105717_86_011621_0_______1_z0.00000.fits
Guardado: GALEXASC_J105933_84_004600_6___1_z0.00000.fits
Guardado: GALEXASC_J105959_49_004408_2___1_z0.00000.fits
Guardado: GALEXMSC_J110022_76_015045_8___1_z0.00000.fits
Guardado: SDSS_J105731_25_014635_4_______1_z0.00000.fits
Guardado: SDSS_J105758_64_013218_0_______1_z0.00000.fits
Guardado: SDSS_J105823_29_015001_3_______1_z0.00000.fits
Guardado: GALEX_2412876205627810558______1_z0.00000.fits
Guardado: WISEA_J030100_28-313732_1______1_z0.00000.fits
Guardado: GAMA_J144050_72_003700_6_______1_z0.00000.fits
Guardado: LCRS_B034346_4-393654__________1_z0.00000.fits
Guardado: LCRS_B103921_1-024555__________1_z0.00000.fits
Guardado: LCRS_B105138_3-051837__________1_z0.00000.fits
Guardado: LCRS_B113550_0-025234

In [ ]:
from astropy.io import fits  # 1. Leer archivos FITS
import numpy as np           # 2. Cálculos numéricos
import matplotlib.pyplot as plt  # 3. Plotting

# 4. Abrir el archivo FITS y extraer datos y cabecera
with fits.open('spectrums_ned/WISEA_J144717_48-060020_7______1_z0.00000.fits') as hdul:
    data   = hdul[0].data      # dato bidimensional: [flujo, cont., error, máscara, …]
    header = hdul[0].header    # metadatos con COEFF0/COEFF1, NAXIS1, etc.

# 5. Obtener número de píxeles espectrales
n_pix = header['NAXIS1']       # 3850 según cabecera

# 6. Construir eje de longitud de onda (vacío, log-linear SDSS)
coeff0 = header['COEFF0']      # 3.5799 → log10(wavelength) en píxel 0
coeff1 = header['COEFF1']      # 1e-4  → incremento en log10(wavelength)
pixels = np.arange(n_pix)      # vector de índices 0…n_pix-1
wave = 10**(coeff0 + coeff1 * pixels)  
#   → λ[i] = 10^(COEFF0 + COEFF1·i) en Ångströms

# 7. Seleccionar el array de flujo (extensión 0)
flux = data[0]                 # unidades 10⁻¹⁷ erg/s/cm²/Å

# 8. Graficar flujo vs longitud de onda
plt.figure()                   
plt.plot(wave, flux)           # trazar curva
plt.xlabel('Longitud de onda (Å)')  
plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
plt.title(f"Espectro de {header.get('NAME','objeto')}, z={header.get('Z', '–')}")
plt.tight_layout()
plt.show()                     # mostrar gráfico

In [ ]:
import os
import pickle
import re
import time
import pandas as pd
from astroquery.utils.tap.core import TapPlus
from astroquery.ipac.ned import Ned
from concurrent.futures import ThreadPoolExecutor, as_completed

# ——————————————————————————————
# 0. Funciones y configuración inicial
# ——————————————————————————————

def sanitize(name):
    """Reemplaza caracteres inválidos para Windows por '_'."""
    return re.sub(r'[^A-Za-z0-9_-]', '_', name)

output_dir    = "spectrums NASA"
os.makedirs(output_dir, exist_ok=True)

pickle_z      = 'extra/redshift_init_ned.pickle'
pickle_meta   = 'extra/all_meta_ned.pickle'
batch_size_in = 10000     # lote inicial ADQL
min_batch     = 10000      # tamaño mínimo tras fallo
max_retries   = 10        # reintentos por lote

all_meta = []             # acumulador de metadatos

# ——————————————————————————————
# 1. Paginación por redshift para metadatos
# ——————————————————————————————
while True:
    # 1.1. Cargar o inicializar z_init
    try:
        with open(pickle_z, 'rb') as f:
            z_init = float(str(pickle.load(f))[:8])
    except FileNotFoundError:
        z_init = 0.0

    current_batch = batch_size_in
    retries       = 0
    success       = False

    # 1.2. Intentos de consulta con reducción de lote
    while not success and retries < max_retries:
        adql = f"""
        SELECT TOP {current_batch}
            prefname, z
        FROM NEDTAP.objdir
        WHERE
            z >= {z_init:.7f}
            AND z < 9.0
            AND n_spectra > 0
        ORDER BY z
        """
        try:
            tap   = TapPlus(url="https://ned.ipac.caltech.edu/tap")
            batch = tap.launch_job(adql).get_results().to_pandas()
            success = True

            if batch.empty:
                break

            all_meta.append(batch)

            # Actualizar z_init  
            last_z = float(str(batch['z'].iloc[-1])[:8])
            if last_z == z_init:
                last_z += 0.000002
            with open(pickle_z, 'wb') as f:
                pickle.dump(last_z, f)

        except Exception:
            current_batch = max(min_batch, current_batch // 2)
            retries += 1
            time.sleep(2)

    if not success or batch.empty:
        break

# 1.4. Guardar metadatos
meta_df = pd.concat(all_meta, ignore_index=True)
meta_df.to_pickle(pickle_meta)
print(f"Metadatos de {len(meta_df)} objetos guardados en '{pickle_meta}'")

# ——————————————————————————————
# 2. Descarga en paralelo con manejo de existentes
# ——————————————————————————————
def download_spectra(prefname, z):
    base = sanitize(prefname)
    saved = []
    try:
        spectra = Ned.get_spectra(prefname, show_progress=False)
    except Exception:
        return saved

    for idx, hdulist in enumerate(spectra, start=1):
        filename = f"{base}_{idx}_z{z:.5f}.fits"
        path     = os.path.join(output_dir, filename)
        if os.path.exists(path):
            print(f"Saltando (ya existe): {filename}")
        else:
            hdulist.writeto(path, overwrite=True)
            print(f"Guardado: {filename}")
        saved.append(path)
    return saved

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {
        executor.submit(download_spectra, row['prefname'], row['z']): ix
        for ix, row in meta_df.iterrows()
    }
    for future in as_completed(futures):
        pass  # los mensajes de guardado/salto se imprimen dentro de download_spectra